In [ ]:
import sys
sys.path.append('..')
import module

In [ ]:
import os
import warnings

os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'


class StderrFilter:
    """Filter untuk menekan log C++ dari MediaPipe/TensorFlow."""

    def __init__(self, original_stderr):
        self.original_stderr = original_stderr

    def write(self, message):
        suppressed_patterns = [
            "FaceBlendshapesGraph acceleration to xnnpack",
            "Feedback manager requires a model",
            "Sets FaceBlendshapesGraph",
            "Created TensorFlow Lite XNNPACK delegate",
        ]
        if message.strip() == "":
            return
        if any(pattern in message for pattern in suppressed_patterns):
            return
        self.original_stderr.write(message)

    def flush(self):
        self.original_stderr.flush()


sys.stderr = StderrFilter(sys.stderr)
warnings.filterwarnings('ignore')

W0000 00:00:1773710374.183589  141068 face_landmarker_graph.cc:174] Sets FaceBlendshapesGraph acceleration to xnnpack by default.
INFO: Created TensorFlow Lite XNNPACK delegate for CPU.
W0000 00:00:1773710374.189838  141070 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.
W0000 00:00:1773710374.202743  141070 inference_feedback_manager.cc:114] Feedback manager requires a model with a single signature inference. Disabling support for feedback tensors.


In [ ]:
import os
from pathlib import Path


BASE_DATASOURCE_PATH=os.path.join(Path.home().as_posix(), "datasets", "anxiety_raw")
BASE_ANNOTATION_PATH=os.path.join(BASE_DATASOURCE_PATH, "annotations.xlsx")

assert os.path.exists(BASE_DATASOURCE_PATH), f"Path tidak ditemukan: {BASE_DATASOURCE_PATH}"
assert os.path.exists(BASE_ANNOTATION_PATH), f"Path tidak ditemukan: {BASE_ANNOTATION_PATH}"

print("Datasource path successfully set.")

Datasource path successfully set.


In [ ]:
import re
import pandas as pd
import numpy as np


dfs = pd.read_excel(BASE_ANNOTATION_PATH, sheet_name=['before', 'after'], engine='openpyxl')

df_before = dfs['before']

# Ekstraksi nomor q dari filepath menggunakan regex
def extract_q_name(path: str):
    m = re.search(r'/q(\d+)(?:/|$)', path)
    return int(m.group(1)) if m else None

# Menambahkan kolom q_num berdasarkan ekstraksi dari filepath
df = df_before.copy()
df['q_num'] = df['filepath'].apply(extract_q_name)

# Mengurutkan data berdasarkan q_num dan subject_name
df_sorted_by_q = df.sort_values(by='q_num', na_position='last')
df_sorted = df.sort_values(by=['subject_name', 'q_num'], na_position='last')

df_sorted.head()

,subject_name,anxiety_level,clip,stage,filepath,q_num
33,aaisyah_nursalsabiil_ni_patriarti,high,q1,before,/home/inadio/datasets/anxiety_raw/before/anxie...,1
34,aaisyah_nursalsabiil_ni_patriarti,high,q2,before,/home/inadio/datasets/anxiety_raw/before/anxie...,2
30,aaisyah_nursalsabiil_ni_patriarti,high,q3,before,/home/inadio/datasets/anxiety_raw/before/anxie...,3
31,aaisyah_nursalsabiil_ni_patriarti,high,q4,before,/home/inadio/datasets/anxiety_raw/before/anxie...,4
32,aaisyah_nursalsabiil_ni_patriarti,high,q5,before,/home/inadio/datasets/anxiety_raw/before/anxie...,5


In [ ]:
import os
import gc
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.notebook import tqdm
from src.apex.modules.v2 import ApexPhaseSpotter


VERSION = ".output-v4"
os.makedirs(f"../{VERSION}", exist_ok=True)

df_annotation = pd.read_csv('../annotations-v2.csv')

spotter = ApexPhaseSpotter()

for index, row in tqdm(df_sorted.iterrows(), total=len(df_sorted), desc="Processing Videos Serial"):
    subject_name = row["subject_name"]
    anxiety_level = row["anxiety_level"]
    video_path = os.path.join(BASE_DATASOURCE_PATH, row["filepath"])
    question_number = Path(video_path).parents[0].name
    
    if subject_name not in df_annotation['subject_name'].values:
        continue

    save_path = f"../{VERSION}/{subject_name}_{question_number}_{anxiety_level}.npy"

    if os.path.exists(save_path):
        continue

    try:
        spotter.process(video_path)
        flow_data = spotter.export_flow_data()
        
        data = {
            "frames": flow_data["frames"],
            "magnitudes": flow_data["magnitudes"],
            "frame_count": flow_data["frame_count"],
            "landmark_detection_rate": flow_data["landmark_detection_rate"],
            "roi_order": flow_data["roi_order"],
        }

        np.save(save_path, data, allow_pickle=True)
        
    except Exception as e:
        tqdm.write(f"ERROR memproses video {video_path}: {e}")
        
    finally:
        # Hapus data sementara sisa proses & jalankan Garbage Collector 
        if 'flow_data' in locals(): del flow_data
        if 'data' in locals(): del data
        gc.collect()

if hasattr(spotter, 'tvl1') and spotter.tvl1:
    spotter.tvl1.close()


Processing Videos Serial:   0%|          | 0/280 [00:00<?, ?it/s]